# LoRA GRPO Fine-Tuning with Kubeflow Trainer and Training Hub on OpenShift AI

This notebook demonstrates how to run **GRPO (Group Relative Policy Optimization)** training on OpenShift AI using a Kubeflow `TrainJob`. GRPO is a reinforcement learning technique that teaches a model to improve its outputs by comparing groups of responses and reinforcing the better ones.

## What is GRPO?

GRPO is a reinforcement learning from verifiable rewards (RLVR) algorithm that:
- Generates multiple candidate responses per prompt
- Scores them with a reward function (e.g. tool-call correctness)
- Uses the group's relative ranking to compute advantage signals
- Updates LoRA adapter weights via policy gradient with group normalization

Each training iteration has two phases:

1. **Rollout phase** — vLLM generates candidate responses, a reward function scores them
2. **Train phase** — Unsloth updates the LoRA adapter weights using the advantage signals

ART time-shares a single GPU between vLLM (inference) and Unsloth (training).

## Training Task: Tool-Call Verification

We use the [Agent-Ark/Toucan-1.5M](https://huggingface.co/datasets/Agent-Ark/Toucan-1.5M) dataset, which contains tool-calling conversations. The reward function verifies that the model produces syntactically correct tool calls with the expected function name and arguments.

## Hardware Requirements

- **1x GPU with 40GB+ VRAM** (A100, H100, or L40S recommended)
- ART manages GPU memory sharing between vLLM and Unsloth via `gpu_memory_utilization`
- The default `gpu_memory_utilization=0.45` reserves 45% for vLLM, leaving the rest for Unsloth

## Setup

First, import the required dependencies.

In [ ]:
# LORA_GRPO support is not yet released in the workbench image.

# Install from midstream until the SDK is pre-installed in the image.

# TODO: replace with midstream branch once the PR is merged, then remove

# this line entirely when the image includes the updated SDK.

%pip install "kubeflow @ git+https://github.com/Fiona-Waters/kubeflow-sdk.git@add-rl-algo" --force-reinstall --quiet

In [ ]:
from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient
from kubeflow.trainer.rhai import TrainingHubAlgorithms, TrainingHubTrainer
from kubernetes import client as k8s

## Authenticate to your OpenShift Cluster

In [ ]:
api_server = "<REPLACE WITH OPENSHIFT SERVER>"

token = "<REPLACE WITH OPENSHIFT TOKEN>"

PVC_NAME = "shared"  # Replace if the shared RWX storage name is different than in the example provided

PVC_PATH = "shared"  # Replace if the shared RWX storage path is different than in the example provided

configuration = k8s.Configuration()

configuration.host = api_server

# Un-comment if your cluster API server uses a self-signed certificate or an un-trusted CA

# configuration.verify_ssl = False

configuration.api_key = {"authorization": f"Bearer {token}"}

api_client = k8s.ApiClient(configuration)

## 1. Configure Training Parameters

Key parameters:

### GRPO Parameters
- **num_iterations**: Number of GRPO iterations (each = rollout + train phase)
- **group_size**: Responses generated per prompt for comparison
- **prompt_batch_size**: Unique prompts sampled per iteration

### LoRA Parameters
- **lora_r**: Rank of the low-rank matrices (higher = more capacity, more memory)
- **lora_alpha**: Scaling factor

### vLLM Parameters
- **gpu_memory_utilization**: Fraction of GPU memory reserved for vLLM inference (rest for Unsloth training)

In [ ]:
# Model and dataset

MODEL_PATH = "Qwen/Qwen3-4B"

DATA_PATH = "Agent-Ark/Toucan-1.5M"

DATA_CONFIG = "Qwen3"


# GRPO hyperparameters

NUM_ITERATIONS = 5  # Number of GRPO iterations (each = rollout + train)

GROUP_SIZE = 4  # Responses generated per prompt for comparison

PROMPT_BATCH_SIZE = 50  # Unique prompts per iteration

N_TRAIN = 200  # Total training examples from dataset

LEARNING_RATE = 1e-5


# LoRA configuration

LORA_R = 16

LORA_ALPHA = 8


# vLLM configuration

GPU_MEMORY_UTILIZATION = 0.45  # Fraction of GPU memory for vLLM (rest for Unsloth)


params = {
    "model_path": MODEL_PATH,
    "data_path": DATA_PATH,
    "data_config": DATA_CONFIG,
    "ckpt_output_dir": f"/mnt/{PVC_PATH}/grpo_output",
    "backend": "art",
    # GRPO hyperparameters
    "num_iterations": NUM_ITERATIONS,
    "group_size": GROUP_SIZE,
    "prompt_batch_size": PROMPT_BATCH_SIZE,
    "n_train": N_TRAIN,
    "learning_rate": LEARNING_RATE,
    # LoRA
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    # vLLM
    "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
}


print("Training Configuration:")

print(f"  Model: {MODEL_PATH}")

print(f"  Dataset: {DATA_PATH} ({DATA_CONFIG})")

print(f"  Iterations: {NUM_ITERATIONS}, Group size: {GROUP_SIZE}")

print(
    f"  Prompts/iter: {PROMPT_BATCH_SIZE}, Rollouts/iter: {PROMPT_BATCH_SIZE * GROUP_SIZE}"
)

print(f"  Training samples: {N_TRAIN}")

print(f"  LoRA Rank: {LORA_R}, Alpha: {LORA_ALPHA}")

print(f"  GPU Memory Utilization: {GPU_MEMORY_UTILIZATION}")

## 2. Training with LORA GRPO and Kubeflow Trainer

Launch a training job via Kubeflow Trainer with configured hyperparameters.

`LORA_GRPO` uses `python` (not `torchrun`) as the entrypoint because ART manages
its own subprocess via `multiprocessing.spawn`. The SDK also wraps the training call
with a `__main__` guard to prevent re-execution in the spawned process.

In [ ]:
backend_cfg = KubernetesBackendConfig(client_configuration=api_client.configuration)

client = TrainerClient(backend_cfg)

### Find the Training Hub ClusterTrainingRuntime

In [ ]:
for runtime in client.list_runtimes():
    if runtime.name == "training-hub":
        th_runtime = runtime

        print("Found runtime: " + str(th_runtime))

### Submit the TrainJob

In [ ]:
from kubeflow.trainer.options.kubernetes import (
    ContainerOverride,
    PodSpecOverride,
    PodTemplateOverride,
    PodTemplateOverrides,
)

cache_root = f"/mnt/{PVC_PATH}/.cache/huggingface"


job_name = client.train(
    trainer=TrainingHubTrainer(
        algorithm=TrainingHubAlgorithms.LORA_GRPO,
        func_args=params,
        env={
            "HF_HOME": cache_root,
            "TRANSFORMERS_ATTN_BACKEND": "sdpa",
        },
        resources_per_node={
            "cpu": 8,
            "memory": "64Gi",
            "nvidia.com/gpu": 1,
        },
    ),
    options=[
        PodTemplateOverrides(
            PodTemplateOverride(
                target_jobs=["node"],
                spec=PodSpecOverride(
                    volumes=[
                        {
                            "name": "work",
                            "persistentVolumeClaim": {"claimName": PVC_NAME},
                        },
                        {
                            "name": "dshm",
                            "emptyDir": {"medium": "Memory"},
                        },
                    ],
                    containers=[
                        ContainerOverride(
                            name="node",
                            volume_mounts=[
                                {
                                    "name": "work",
                                    "mountPath": f"/mnt/{PVC_PATH}",
                                    "readOnly": False,
                                },
                                {
                                    "name": "dshm",
                                    "mountPath": "/dev/shm",
                                },
                            ],
                        ),
                    ],
                ),
            )
        )
    ],
    runtime=th_runtime,
)


print(job_name)

### Follow job logs

In [ ]:
# Follow job logs

logs = client.get_job_logs(job_name, follow=True)

for line in logs:
    print(line)

## 3. Inspect Results

After training completes, check the metrics file and training results on the PVC.

GRPO writes two lines per iteration to `training_metrics.jsonl`:
- **Rollout phase**: `mean_reward`, `full_match_rate`
- **Train phase**: `loss`, `grad_norm`, `entropy`

In [ ]:
import json
import os

OUTPUT_DIR = f"/opt/app-root/src/{PVC_PATH}/grpo_output"


# Read training metrics (two lines per iteration: rollout + train)

metrics_file = os.path.join(OUTPUT_DIR, "training_metrics.jsonl")

if os.path.exists(metrics_file):
    print("Training metrics:")

    print("=" * 80)

    with open(metrics_file) as f:
        for line in f:
            entry = json.loads(line)

            phase = entry.get("phase", "unknown")

            step = entry.get("step", "?")

            if phase == "rollout":
                print(
                    f"Step {step} [rollout]  mean_reward={entry.get('mean_reward', 'N/A'):.4f}  "
                    f"full_match_rate={entry.get('full_match_rate', 'N/A'):.4f}"
                )

            elif phase == "train":
                print(
                    f"Step {step} [train]    loss={entry.get('loss', 'N/A')}  "
                    f"grad_norm={entry.get('grad_norm', 'N/A')}  "
                    f"entropy={entry.get('entropy', 'N/A')}"
                )

else:
    print("No metrics file yet -- training may not have started.")

In [ ]:
# Read training results summary

results_file = os.path.join(OUTPUT_DIR, "training_results.json")

if os.path.exists(results_file):
    with open(results_file) as f:
        results = json.load(f)

    print("Training results:")

    print(json.dumps(results, indent=2, default=str))

else:
    print("No results file yet -- training may not have completed.")

## 4. Plot Reward Curve

In [ ]:
import json
import os

metrics_file = os.path.join(OUTPUT_DIR, "training_metrics.jsonl")

if not os.path.exists(metrics_file):
    print("No metrics file found.")

else:
    steps, rewards, match_rates = [], [], []

    with open(metrics_file) as f:
        for line in f:
            entry = json.loads(line)

            if entry.get("phase") == "rollout":
                steps.append(entry["step"])

                rewards.append(entry["mean_reward"])

                match_rates.append(entry["full_match_rate"])

    try:
        import matplotlib.pyplot as plt

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

        ax1.plot(steps, rewards, marker="o")

        ax1.set_xlabel("Iteration")

        ax1.set_ylabel("Mean Reward")

        ax1.set_title("GRPO Training: Mean Reward")

        ax1.grid(True, alpha=0.3)

        ax2.plot(steps, match_rates, marker="o", color="tab:orange")

        ax2.set_xlabel("Iteration")

        ax2.set_ylabel("Full Match Rate")

        ax2.set_title("GRPO Training: Full Match Rate")

        ax2.grid(True, alpha=0.3)

        plt.tight_layout()

        plt.savefig(os.path.join(OUTPUT_DIR, "reward_curve.png"), dpi=150)

        plt.show()

        print(f"Plot saved to {OUTPUT_DIR}/reward_curve.png")

    except ImportError:
        print("matplotlib not installed -- printing raw values instead.")

        for s, r, m in zip(steps, rewards, match_rates, strict=True):
            print(f"  Step {s}: reward={r:.4f}, match_rate={m:.4f}")

## 5. Test the Trained Model

Load the fine-tuned LoRA adapter from the shared PVC and test whether the model generates better tool calls after GRPO training.

> **Note:** This section requires a GPU on the workbench to run inference.

In [ ]:
from pathlib import Path

from unsloth import FastLanguageModel

# Find the latest checkpoint on the shared PVC

art_dir = Path(OUTPUT_DIR) / ".art"

checkpoints_dirs = sorted(art_dir.rglob("checkpoints"))

if not checkpoints_dirs:
    raise FileNotFoundError(f"No checkpoints found under {art_dir}")

latest_ckpt = sorted(checkpoints_dirs[0].iterdir())[-1]

print(f"Loading checkpoint: {latest_ckpt}")


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(latest_ckpt),
    max_seq_length=2048,
    load_in_4bit=False,
)

FastLanguageModel.for_inference(model)


print("Fine-tuned model loaded and ready for inference.")

In [ ]:
import torch


def generate_response(messages, max_tokens=512):
    """Generate a response from the fine-tuned model."""

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )

    return response

In [ ]:
test_prompts = [
    {
        "description": "Weather lookup",
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant with access to tools. "
                    "Available tools: get_weather(location: str, unit: str = 'celsius') -> dict"
                ),
            },
            {"role": "user", "content": "What's the weather like in Dublin?"},
        ],
    },
    {
        "description": "Calculator",
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant with access to tools. "
                    "Available tools: calculate(expression: str) -> float"
                ),
            },
            {"role": "user", "content": "What is 1547 * 23 + 89?"},
        ],
    },
    {
        "description": "File search",
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant with access to tools. "
                    "Available tools: search_files(query: str, directory: str = '.', file_type: str = None) -> list"
                ),
            },
            {
                "role": "user",
                "content": "Find all Python files that contain 'import torch'",
            },
        ],
    },
]

print("Testing fine-tuned model on tool-call generation:")
print("=" * 60)

for test in test_prompts:
    print(f"\nTest: {test['description']}")
    print(f"User: {test['messages'][-1]['content']}")
    response = generate_response(test["messages"])
    print(f"Model: {response}")
    print("-" * 60)

## 6. Cleanup

In [ ]:
client.delete_job(job_name)

print(f"TrainJob '{job_name}' deleted.")